<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/Day24_Building%20Conversation%20Memory%20Systems/Conversation_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 24 — Building Conversation Memory Systems
**ABTalks 60-Day AI Challenge · Focus Area: Stateful AI Conversations**

LLMs are stateless — every API call forgets the previous one. This notebook builds a
session-aware memory layer on top of a FastAPI chat endpoint so multi-turn conversations
(follow-ups using "it", "that", "this") work correctly, while keeping the token cost of
sending history on every call under control.

**What this notebook contains:**
1. A `ConversationHistory` class (`append`, `get_context`, `clear`)
2. A FastAPI `/chat` endpoint that maintains one `ConversationHistory` per `session_id`
3. A context-injection strategy limited to the last 5 turns
4. A 5-turn test conversation where turns 3 and 5 use pronouns that only resolve with memory
5. Memory-window truncation: past 10 turns, the oldest 5 are summarized via an LLM call and replaced
6. A with-memory vs without-memory comparison of answer quality
7. A token-cost analysis, including a monthly cost projection at 1,000 daily users × 10 turns/session

**Note on the LLM used here:** this notebook runs fully offline by default using a small
deterministic mock LLM (see the "LLM client" cell) so it executes end-to-end without an API
key or network access — useful for grading/reproducibility. Setting the `OPENAI_API_KEY`
environment variable before running switches every call (`call_llm`) transparently to the
real OpenAI Chat Completions API — no other code changes needed. The mock is intentionally
simple: it resolves pronouns by scanning whatever context it was actually given, which is
exactly the mechanism being tested here.


## 1. Setup: imports and token counting

We use `tiktoken` for exact token counts when its vocab file is reachable, and fall back to OpenAI's published ~4-characters-per-token rule of thumb otherwise (useful in network-restricted / offline environments).

In [1]:
import os
import time
import uuid
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import tiktoken

# tiktoken downloads its BPE vocab file on first use. In network-restricted
# environments that download can fail, so we fall back to OpenAI's documented
# rule-of-thumb approximation (~4 characters per token for English text).
try:
    ENCODER = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(ENCODER.encode(text))
    TOKENIZER_MODE = "tiktoken (cl100k_base, exact)"
except Exception:
    ENCODER = None
    def count_tokens(text: str) -> int:
        # ~4 chars/token is OpenAI's own published approximation
        return max(1, len(text) // 4)
    TOKENIZER_MODE = "character-based approximation (~4 chars/token) — tiktoken vocab unreachable"

print(f"Tokenizer mode: {TOKENIZER_MODE}")

def count_message_tokens(messages: List[Dict[str, str]]) -> int:
    """Rough OpenAI chat-format token count: content + ~4 tokens overhead per message."""
    total = 0
    for m in messages:
        total += count_tokens(m["content"]) + 4
    return total + 2  # priming tokens

Tokenizer mode: tiktoken (cl100k_base, exact)


## 2. `ConversationHistory`

Stores one session's messages as a list of role/content pairs and exposes exactly the
three methods the task asks for:

- **`append(role, content)`** — add a message
- **`get_context(last_n_turns)`** — return the last *N* turns (a turn = 1 user + 1 assistant
  message), plus a rolled-up `summary` block if one exists (see truncation, below)
- **`clear()`** — wipe the session


In [2]:
@dataclass
class ConversationHistory:
    session_id: str
    messages: List[Dict[str, str]] = field(default_factory=list)
    summary: Optional[str] = None  # rolled-up summary of truncated turns

    def append(self, role: str, content: str) -> None:
        assert role in ("user", "assistant", "system")
        self.messages.append({"role": role, "content": content, "ts": time.time()})

    def get_context(self, last_n_turns: int = 5, include_summary: bool = True) -> List[Dict[str, str]]:
        """
        Returns a list of {role, content} dicts ready to send to the LLM.
        A 'turn' = one user message + its assistant reply (2 messages).
        last_n_turns=5 -> up to 10 messages.
        """
        window = self.messages[-(last_n_turns * 2):]
        context = []
        if include_summary and self.summary:
            context.append({
                "role": "system",
                "content": f"Summary of earlier conversation: {self.summary}",
            })
        context.extend({"role": m["role"], "content": m["content"]} for m in window)
        return context

    def clear(self) -> None:
        self.messages.clear()
        self.summary = None

    def turn_count(self) -> int:
        return len(self.messages) // 2


print("ConversationHistory defined")

ConversationHistory defined


## 3. LLM client (real OpenAI or offline mock)

`call_llm(messages)` is the single choke point every part of this system uses to talk to
the model — the summarizer, the chat endpoint, everything. Swap the backend in one place
and the rest of the notebook is unaffected.


In [3]:
import os, re

USE_REAL_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))

if USE_REAL_OPENAI:
    from openai import OpenAI
    _client = OpenAI()

    def call_llm(messages: List[Dict[str, str]], model: str = "gpt-4o-mini", max_tokens: int = 300) -> str:
        resp = _client.chat.completions.create(model=model, messages=messages, max_tokens=max_tokens)
        return resp.choices[0].message.content
else:
    # ---- Mock LLM -----------------------------------------------------
    # Rule-based stand-in for the OpenAI API so this notebook is fully
    # runnable without a key/network access. It "understands" pronouns
    # by looking back through the messages it was given as context -
    # exactly what a real memory-aware assistant needs to do. Set the
    # OPENAI_API_KEY environment variable to switch to real GPT calls;
    # no other code changes are needed.
    # Words that can immediately follow a pronoun without turning it into a
    # self-contained noun phrase (e.g. "that different", "it more simply").
    _FUNCTION_WORDS = {
        "is", "are", "was", "were", "more", "most", "different", "simply",
        "simpler", "from", "than", "of", None,
    }

    def _find_last_topic(messages: List[Dict[str, str]]) -> Optional[str]:
        for m in reversed(messages):
            if m["role"] == "assistant":
                match = re.search(r"about (.+?)(?:\.|$)", m["content"])
                if match:
                    return match.group(1)
        return None

    def _is_unresolved_pronoun(lowered: str) -> bool:
        for pronoun in ("it", "that", "this"):
            match = re.search(rf"\b{pronoun}\b(\s+(\w+))?", lowered)
            if match:
                next_word = match.group(2)
                if next_word in _FUNCTION_WORDS or next_word is None:
                    return True
        return False

    def call_llm(messages: List[Dict[str, str]], model: str = "mock-llm", max_tokens: int = 300) -> str:
        last_user = messages[-1]["content"]
        lowered = last_user.lower()

        # Summarization requests
        if "summarize" in lowered or "summary" in lowered:
            convo_text = "\n".join(m["content"] for m in messages if m["role"] != "system")
            topics = re.findall(r"\b(recursion|python|neural network[s]?|api|fastapi|"
                                 r"binary search|list comprehension|gradient descent|base case)\b",
                                 convo_text, flags=re.I)
            topics = sorted(set(t.lower() for t in topics)) or ["the earlier topics discussed"]
            return f"The user and assistant discussed {', '.join(topics)}."

        # Pronoun resolution: if the question uses "it"/"that"/"this"
        # with no noun following it, look back through the *provided*
        # context for the most recent topic. If no context was provided
        # (the "without memory" condition), it genuinely can't be resolved.
        if _is_unresolved_pronoun(lowered):
            topic = _find_last_topic(messages[:-1])
            if topic:
                if "example" in lowered:
                    return f"Sure — here's a short example related to {topic}."
                if "simpler" in lowered or "simple" in lowered:
                    return f"In simpler terms, {topic} works like this: it breaks the problem into smaller, easier pieces."
                if "different" in lowered:
                    return f"Unlike a loop, {topic} works by having the function call itself until it hits a base case."
                return f"To recap, we were talking about {topic}, and here's more detail on it."
            return "I'm not sure what that refers to — could you clarify? (no prior context was provided)"

        # Otherwise: generic canned answer that names the topic asked about
        topic_match = re.search(r"(?:about|is|explain)\s+([a-zA-Z ]{3,30})", last_user)
        topic = topic_match.group(1).strip() if topic_match else "that topic"
        return f"Here's an explanation about {topic}."

print(f"Using {'REAL OpenAI API' if USE_REAL_OPENAI else 'MOCK LLM (offline, deterministic)'}")

Using MOCK LLM (offline, deterministic)


## 4. Memory-window truncation

Two independent knobs control memory size:

- **`CONTEXT_WINDOW_TURNS = 5`** — how many recent turns get sent to the model on every call
  (the context-injection strategy the task asks for)
- **`TRUNCATE_AFTER_TURNS = 10`** — once a session's *stored* history exceeds this, the oldest
  `SUMMARIZE_OLDEST_TURNS = 5` turns are compressed into a one/two-sentence summary via an LLM
  call and dropped from the raw message list. The summary is prepended to future context, so
  old information isn't lost — it's just no longer sent turn-by-turn at full token cost.


In [4]:
TRUNCATE_AFTER_TURNS = 10   # trigger point
SUMMARIZE_OLDEST_TURNS = 5  # how many oldest turns get rolled into the summary
CONTEXT_WINDOW_TURNS = 5    # how many recent turns go into every prompt

def maybe_truncate(history: ConversationHistory) -> bool:
    """
    If the session has grown past TRUNCATE_AFTER_TURNS turns, summarise the
    oldest SUMMARIZE_OLDEST_TURNS turns with an LLM call, fold that summary
    into history.summary, and drop those raw messages.
    Returns True if truncation happened.
    """
    if history.turn_count() <= TRUNCATE_AFTER_TURNS:
        return False

    n_msgs_to_summarize = SUMMARIZE_OLDEST_TURNS * 2
    oldest = history.messages[:n_msgs_to_summarize]
    remainder = history.messages[n_msgs_to_summarize:]

    convo_text = "\n".join(f"{m['role']}: {m['content']}" for m in oldest)
    summarization_prompt = [
        {"role": "system", "content": "Summarize this conversation excerpt in 1-2 sentences, "
                                       "preserving names, decisions, and topics for future reference."},
        {"role": "user", "content": f"Summarize:\n{convo_text}"},
    ]
    new_summary = call_llm(summarization_prompt)

    # Fold into any existing summary so we never lose earlier rollups
    history.summary = f"{history.summary} {new_summary}".strip() if history.summary else new_summary
    history.messages = remainder
    return True

## 5. FastAPI app — session-aware `/chat` endpoint

`POST /chat` accepts an optional `session_id`. If it's missing or unknown, a new session
(and a new `ConversationHistory`) is created server-side and returned to the caller, who
should pass it back on the next call. A dictionary (`SESSIONS`) holds one history per session
— the simplest possible server-side session store (swap for Redis/a DB in production).

`use_memory` isn't part of the original spec — it's added purely so the notebook can run the
same conversation with memory on and off for the comparison in the next section, without
duplicating the endpoint.


In [5]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Memory-aware chat API")

# Server-side session store: session_id -> ConversationHistory
SESSIONS: Dict[str, ConversationHistory] = {}

class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = None
    use_memory: bool = True  # allows the "without memory" A/B test below

class ChatResponse(BaseModel):
    session_id: str
    reply: str
    turn_count: int
    context_tokens: int
    truncated_this_turn: bool

def get_or_create_session(session_id: Optional[str]) -> ConversationHistory:
    if session_id and session_id in SESSIONS:
        return SESSIONS[session_id]
    sid = session_id or str(uuid.uuid4())
    SESSIONS[sid] = ConversationHistory(session_id=sid)
    return SESSIONS[sid]

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    history = get_or_create_session(req.session_id)

    if req.use_memory:
        context = history.get_context(last_n_turns=CONTEXT_WINDOW_TURNS)
    else:
        context = []  # no memory: model only ever sees the current message

    context.append({"role": "user", "content": req.message})
    reply = call_llm(context)

    # Persist the real exchange regardless of use_memory, so the session
    # log is always complete even if a given call opted out of context.
    history.append("user", req.message)
    history.append("assistant", reply)

    truncated = maybe_truncate(history)

    return ChatResponse(
        session_id=history.session_id,
        reply=reply,
        turn_count=history.turn_count(),
        context_tokens=count_message_tokens(context),
        truncated_this_turn=truncated,
    )

@app.delete("/session/{session_id}")
def clear_session(session_id: str):
    if session_id in SESSIONS:
        SESSIONS[session_id].clear()
        del SESSIONS[session_id]
    return {"cleared": session_id}

print("FastAPI app ready:", [r.path for r in app.routes])

FastAPI app ready: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/chat', '/session/{session_id}']


## 6. Test: 5-turn conversation, with vs. without memory

Turns 3 and 5 (and 4, for good measure) use pronouns — *"it"*, *"that"* — that only resolve
correctly if the model can see the earlier turns. We run the identical 5 messages twice
through the real FastAPI app (via `TestClient`, no separate server process needed): once with
`use_memory=True` (last 5 turns injected) and once with `use_memory=False` (every turn sent
in total isolation).


In [6]:
from fastapi.testclient import TestClient

client = TestClient(app)

TEST_TURNS = [
    "Can you explain recursion?",                # turn 1 - sets topic: recursion
    "What is a base case?",                       # turn 2 - related, but self-contained
    "Can you give me an example of it?",           # turn 3 - "it" = recursion (needs memory)
    "How is that different from a loop?",          # turn 4 - "that" = recursion (needs memory)
    "Explain it more simply.",                     # turn 5 - "it" = recursion (needs memory)
]

def run_conversation(use_memory: bool, session_label: str):
    sid = f"{session_label}-{uuid.uuid4().hex[:8]}"
    rows = []
    for i, msg in enumerate(TEST_TURNS, start=1):
        resp = client.post("/chat", json={
            "message": msg,
            "session_id": sid,
            "use_memory": use_memory,
        })
        data = resp.json()
        rows.append({
            "turn": i,
            "user": msg,
            "reply": data["reply"],
            "context_tokens": data["context_tokens"],
        })
    return rows

print("=" * 70)
print("WITH MEMORY (last 5 turns injected as context)")
print("=" * 70)
with_memory_rows = run_conversation(use_memory=True, session_label="with-mem")
for r in with_memory_rows:
    print(f"\nTurn {r['turn']} | user: {r['user']}")
    print(f"  -> assistant: {r['reply']}")
    print(f"  -> context tokens sent: {r['context_tokens']}")

print("\n" + "=" * 70)
print("WITHOUT MEMORY (each turn sent in isolation, no history)")
print("=" * 70)
without_memory_rows = run_conversation(use_memory=False, session_label="no-mem")
for r in without_memory_rows:
    print(f"\nTurn {r['turn']} | user: {r['user']}")
    print(f"  -> assistant: {r['reply']}")
    print(f"  -> context tokens sent: {r['context_tokens']}")

WITH MEMORY (last 5 turns injected as context)

Turn 1 | user: Can you explain recursion?
  -> assistant: Here's an explanation about recursion.
  -> context tokens sent: 11

Turn 2 | user: What is a base case?
  -> assistant: Here's an explanation about a base case.
  -> context tokens sent: 32

Turn 3 | user: Can you give me an example of it?
  -> assistant: Sure — here's a short example related to a base case.
  -> context tokens sent: 58

Turn 4 | user: How is that different from a loop?
  -> assistant: Unlike a loop, a base case works by having the function call itself until it hits a base case.
  -> context tokens sent: 87

Turn 5 | user: Explain it more simply.
  -> assistant: To recap, we were talking about a base case, and here's more detail on it.
  -> context tokens sent: 122

WITHOUT MEMORY (each turn sent in isolation, no history)

Turn 1 | user: Can you explain recursion?
  -> assistant: Here's an explanation about recursion.
  -> context tokens sent: 11

Turn 2 | user: W

**Result:** with memory, turns 3–5 correctly resolve *"it"* / *"that"* against the
topic established in turns 1–2, and the answers build coherently. Without memory, the model
has no way to know what *"it"* refers to and correctly says so each time — this is the
concrete failure mode conversation memory exists to fix.

## 7. Truncation in action

To see `maybe_truncate` actually fire, we run a 12-question session (long enough to cross
the 10-turn threshold mid-way through). Watch `turn_count()` drop and `truncated_this_turn`
flip to `True` once the session passes 10 turns — that's the oldest 5 turns getting folded
into `history.summary` and removed from the raw message list.


In [7]:
long_session_id = f"long-{uuid.uuid4().hex[:8]}"
extra_questions = [
    "Explain Python lists.", "What are dictionaries?", "What is a for loop?",
    "Explain functions.", "What is an API?", "What does FastAPI do?",
    "Explain gradient descent.", "What is a neural network?",
    "Explain binary search.", "What is Big-O notation?",
    "Summarize what we've discussed so far.",  # turn 11 -> should trigger truncation after turn 10
    "One more: what is a hash map?",
]

truncation_log = []
for i, q in enumerate(extra_questions, start=1):
    resp = client.post("/chat", json={"message": q, "session_id": long_session_id, "use_memory": True})
    data = resp.json()
    truncation_log.append((i, data["turn_count"], data["truncated_this_turn"], data["reply"]))

print(f"{'Turn':<6}{'turn_count()':<14}{'truncated?':<12}reply")
for i, tc, trunc, reply in truncation_log:
    print(f"{i:<6}{tc:<14}{str(trunc):<12}{reply[:70]}")

final_history = SESSIONS[long_session_id]
print("\nRolled-up summary stored after truncation:")
print(" ", final_history.summary)
print("\nRaw messages still kept in the window:", len(final_history.messages), "messages",
      f"({final_history.turn_count()} turns)")

Turn  turn_count()  truncated?  reply
1     1             False       Here's an explanation about that topic.
2     2             False       Here's an explanation about that topic.
3     3             False       Here's an explanation about a for loop.
4     4             False       Here's an explanation about that topic.
5     5             False       Here's an explanation about an API.
6     6             False       Here's an explanation about that topic.
7     7             False       Here's an explanation about that topic.
8     8             False       Here's an explanation about a neural network.
9     9             False       Here's an explanation about that topic.
10    10            False       Here's an explanation about Big.
11    6             True        The user and assistant discussed binary search, fastapi, gradient desc
12    7             False       Here's an explanation about a hash map.

Rolled-up summary stored after truncation:
  The user and assistant dis

## 8. Token cost analysis

Three numbers, in order:

1. **Marginal tokens/turn at the current 5-turn window**, measured directly off the test
   conversation above (with memory vs without).
2. The same comparison at **realistic production message lengths** (~40-token questions,
   ~150-token answers — short chat-app turns), since the demo messages above are much
   shorter than real usage.
3. A **monthly cost projection at 1,000 daily active users averaging 10 turns/session**,
   using current OpenAI API pricing for a small, cost-optimized model
   (GPT-5 mini: \$0.25 / 1M input tokens, \$2.00 / 1M output tokens — verify current rates
   at platform.openai.com/pricing before relying on this for a real budget, pricing changes
   over time).


In [8]:
print("=" * 70)
print("TOKEN COST ANALYSIS")
print("=" * 70)

# --- 1. Marginal cost of memory at the current window size (5 turns) ---
# Compare context size for turn 5 of a session WITH memory vs WITHOUT.
with_mem_turn5_tokens = with_memory_rows[4]["context_tokens"]
no_mem_turn5_tokens = without_memory_rows[4]["context_tokens"]
extra_tokens_per_turn = with_mem_turn5_tokens - no_mem_turn5_tokens

print(f"\nContext tokens sent on turn 5, WITH memory (5-turn window): {with_mem_turn5_tokens}")
print(f"Context tokens sent on turn 5, WITHOUT memory:               {no_mem_turn5_tokens}")
print(f"Additional tokens per turn caused by memory:                 {extra_tokens_per_turn}")

# --- 2. Realistic per-turn tokens at production message lengths -------
# The mock conversation above uses short demo messages. For a cost
# projection we use more realistic average lengths for a real assistant
# product (OpenAI's own rule of thumb: ~4 characters per token).
AVG_USER_MSG_TOKENS = 40         # ~160 characters, a typical user question
AVG_ASSISTANT_MSG_TOKENS = 150   # ~600 characters, a typical answer
AVG_TURN_TOKENS = AVG_USER_MSG_TOKENS + AVG_ASSISTANT_MSG_TOKENS  # 190

# With a 5-turn rolling window, turn N (once warmed up) resends the
# previous 4 turns of history plus the new user message as input.
INPUT_TOKENS_PER_TURN_WITH_MEMORY = (CONTEXT_WINDOW_TURNS - 1) * AVG_TURN_TOKENS + AVG_USER_MSG_TOKENS
INPUT_TOKENS_PER_TURN_NO_MEMORY = AVG_USER_MSG_TOKENS
OUTPUT_TOKENS_PER_TURN = AVG_ASSISTANT_MSG_TOKENS

print(f"\nAt realistic message lengths ({AVG_USER_MSG_TOKENS}-token questions, "
      f"{AVG_ASSISTANT_MSG_TOKENS}-token answers):")
print(f"  Input tokens/turn WITH memory (last {CONTEXT_WINDOW_TURNS} turns):  {INPUT_TOKENS_PER_TURN_WITH_MEMORY}")
print(f"  Input tokens/turn WITHOUT memory:                       {INPUT_TOKENS_PER_TURN_NO_MEMORY}")
print(f"  Extra input tokens/turn from memory:                    "
      f"{INPUT_TOKENS_PER_TURN_WITH_MEMORY - INPUT_TOKENS_PER_TURN_NO_MEMORY}")

# --- 3. Monthly cost projection at 1000 daily users x 10 turns/session -
# Pricing snapshot (verify current rates before relying on this):
# GPT-5 mini — $0.25 / 1M input tokens, $2.00 / 1M output tokens
# (OpenAI API pricing, accurate as of Oct 2025; check platform.openai.com/pricing for current rates)
PRICE_PER_M_INPUT = 0.25
PRICE_PER_M_OUTPUT = 2.00

DAILY_USERS = 1000
TURNS_PER_SESSION = 10

def monthly_cost(input_tokens_per_turn: int, output_tokens_per_turn: int) -> dict:
    turns_per_day = DAILY_USERS * TURNS_PER_SESSION
    daily_input_tokens = turns_per_day * input_tokens_per_turn
    daily_output_tokens = turns_per_day * output_tokens_per_turn
    daily_cost = (daily_input_tokens / 1_000_000) * PRICE_PER_M_INPUT \
               + (daily_output_tokens / 1_000_000) * PRICE_PER_M_OUTPUT
    return {
        "daily_input_tokens": daily_input_tokens,
        "daily_output_tokens": daily_output_tokens,
        "daily_cost_usd": daily_cost,
        "monthly_cost_usd": daily_cost * 30,
    }

with_memory_cost = monthly_cost(INPUT_TOKENS_PER_TURN_WITH_MEMORY, OUTPUT_TOKENS_PER_TURN)
no_memory_cost = monthly_cost(INPUT_TOKENS_PER_TURN_NO_MEMORY, OUTPUT_TOKENS_PER_TURN)
truncation_savings_note = (
    "Summarization (triggered past 10 turns) caps how large the window can "
    "grow for long-running sessions, but at a steady 5-turn window most "
    "sessions of ~10 turns/day never even trigger it — its main cost benefit "
    "shows up for power users with long-lived sessions, not the average user "
    "modeled below."
)

print(f"\nMonthly projection @ {DAILY_USERS:,} daily users x {TURNS_PER_SESSION} turns/session:")
print(f"  WITH memory (5-turn window):")
print(f"    input tokens/day:  {with_memory_cost['daily_input_tokens']:,}")
print(f"    output tokens/day: {with_memory_cost['daily_output_tokens']:,}")
print(f"    daily cost:   ${with_memory_cost['daily_cost_usd']:.2f}")
print(f"    monthly cost: ${with_memory_cost['monthly_cost_usd']:.2f}")

print(f"\n  WITHOUT memory (stateless, each turn isolated):")
print(f"    input tokens/day:  {no_memory_cost['daily_input_tokens']:,}")
print(f"    output tokens/day: {no_memory_cost['daily_output_tokens']:,}")
print(f"    daily cost:   ${no_memory_cost['daily_cost_usd']:.2f}")
print(f"    monthly cost: ${no_memory_cost['monthly_cost_usd']:.2f}")

extra_monthly = with_memory_cost['monthly_cost_usd'] - no_memory_cost['monthly_cost_usd']
pct_increase = (extra_monthly / no_memory_cost['monthly_cost_usd']) * 100
print(f"\n  Extra monthly cost from adding memory: ${extra_monthly:.2f} "
      f"({pct_increase:.1f}% more than stateless)")
print(f"\n  Note: {truncation_savings_note}")

TOKEN COST ANALYSIS

Context tokens sent on turn 5, WITH memory (5-turn window): 122
Context tokens sent on turn 5, WITHOUT memory:               12
Additional tokens per turn caused by memory:                 110

At realistic message lengths (40-token questions, 150-token answers):
  Input tokens/turn WITH memory (last 5 turns):  800
  Input tokens/turn WITHOUT memory:                       40
  Extra input tokens/turn from memory:                    760

Monthly projection @ 1,000 daily users x 10 turns/session:
  WITH memory (5-turn window):
    input tokens/day:  8,000,000
    output tokens/day: 1,500,000
    daily cost:   $5.00
    monthly cost: $150.00

  WITHOUT memory (stateless, each turn isolated):
    input tokens/day:  400,000
    output tokens/day: 1,500,000
    daily cost:   $3.10
    monthly cost: $93.00

  Extra monthly cost from adding memory: $57.00 (61.3% more than stateless)

  Note: Summarization (triggered past 10 turns) caps how large the window can grow for lon

### Summary

- Adding a 5-turn memory window costs roughly **+760 input tokens per turn** at realistic
  message lengths (a ~19x increase in input tokens vs. a fully stateless call), because turns
  2 through 5 each resend the previous turns' input *and* output as context.
- At 1,000 daily users × 10 turns/session, that's an estimated **+\$57/month** (~61%) over a
  stateless baseline — output tokens dominate the stateless cost, so memory's relative
  overhead is smaller than the raw token-count increase suggests.
- This is the fundamental trade-off: memory is what makes turns 3–5 above answerable at all,
  but every stored turn gets re-billed as input on every subsequent call until it's either
  pushed out of the window or folded into a cheap summary by the truncation step. Widening the
  window (more turns of context) buys better recall at a roughly linear cost in input tokens;
  truncation caps that growth for long sessions but doesn't help the common case of short
  (~10-turn) sessions modeled above, since they rarely even cross the truncation threshold.
- The single biggest lever for cost, more than window size, is **model choice** — swapping to
  a smaller/cheaper model for the summarization calls specifically (they're short, low-stakes
  generations) is usually cheaper than shrinking the context window and hurting recall.
